# Databricks notebook source
##### Gold Layer Data Load: Stock-to-Cash Reconciliation Fact (`gold_fact_stock_cash_recon`)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

###### 1. Initialize Gold Schema & Target Table Name

In [0]:
catalog_name = "glrecon"
gold_table_name = f"{catalog_name}.gold.gold_fact_stock_cash_recon"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")

###### Tolerance threshold for minor rounding or bank fee variances

In [0]:
TOLERANCE_AMOUNT = 5.00

###### 2. Read Silver Stock & Cash Transactions

In [0]:
df_stock = spark.table(f"{catalog_name}.silver.silver_stock_transactions")
df_cash = spark.table(f"{catalog_name}.silver.silver_cash_transactions")

###### 3. Perform Reconciliation & Variance Classification

In [0]:
df_recon_staged = (
    df_stock.alias("s")
    .join(
        df_cash.alias("c"),
        F.col("s.StockTransactionId") == F.col("c.StockTransactionId"),
        "left"
    )
    .select(
        F.col("s.StockTransactionId"),
        F.col("c.CashTransactionId"),
        F.col("s.ProductId").cast("long").alias("ProductId"),
        F.col("s.TransactionDate").cast("date").alias("StockDate"),
        F.col("c.TransactionDate").cast("date").alias("CashDate"),
        F.col("s.TransDescription").alias("StockTransDescription"),
        F.col("c.TransDescription").alias("CashTransDescription"),
        F.col("s.Quantity").cast("int").alias("Quantity"),
        F.col("s.UnitPrice").cast("decimal(12,2)").alias("UnitPrice"),
        (F.col("s.Quantity") * F.col("s.UnitPrice")).cast("decimal(14,2)").alias("ExpectedStockValue"),
        F.coalesce(F.col("c.CashTransAmount"), F.lit(0)).cast("decimal(14,2)").alias("ActualCashAmount")
    )
    .withColumn(
        "VarianceAmount",
        (F.col("ExpectedStockValue") - F.col("ActualCashAmount")).cast("decimal(14,2)")
    )
    .withColumn(
        "SettlementLagDays",
        F.datediff(F.col("CashDate"), F.col("StockDate"))
    )
    .withColumn(
        "ReconciliationStatus",
        F.when(F.col("CashTransactionId").isNull(), "UNMATCHED MISSING CASH")
        .when(
            ((F.col("StockTransDescription") == "PURCHASE") & (F.col("CashTransDescription") != "SUPPLIER PAYMENT")) |
            ((F.col("StockTransDescription") == "SALE") & (F.col("CashTransDescription") != "CUSTOMER RECEIPT")),
            "DIRECTION MISMATCH"
        )
        .when(F.abs(F.col("VarianceAmount")) == 0, "MATCHED EXACT")
        .when(F.abs(F.col("VarianceAmount")) <= TOLERANCE_AMOUNT, "MATCHED WITHIN TOLERANCE")
        .otherwise("AMOUNT VARIANCE EXCEPTION")
    )
    .withColumn("GoldProcessedDate", F.current_timestamp())
)

###### 4. Upsert into Gold Reconciliation Fact Table (`MERGE`)

In [0]:
if not spark.catalog.tableExists(gold_table_name):
    print(f"Table {gold_table_name} does not exist. Initializing Gold fact table...")
    
    df_recon_staged.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    print("Gold reconciliation fact table successfully created and populated.")

else:
    print(f"Table {gold_table_name} exists. Proceeding with MERGE pipeline...")
    
    gold_table = DeltaTable.forName(spark, gold_table_name)

    (
        gold_table.alias("target")
        .merge(
            df_recon_staged.alias("source"),
            "target.StockTransactionId = source.StockTransactionId"
        )
        .whenMatchedUpdate(
            condition="""
                NOT (target.CashTransactionId <=> source.CashTransactionId) OR
                NOT (target.ActualCashAmount <=> source.ActualCashAmount) OR
                NOT (target.VarianceAmount <=> source.VarianceAmount) OR
                NOT (target.ReconciliationStatus <=> source.ReconciliationStatus) OR
                NOT (target.CashDate <=> source.CashDate)
            """,
            set={
                "CashTransactionId": "source.CashTransactionId",
                "ProductId": "source.ProductId",
                "StockDate": "source.StockDate",
                "CashDate": "source.CashDate",
                "StockTransDescription": "source.StockTransDescription",
                "CashTransDescription": "source.CashTransDescription",
                "Quantity": "source.Quantity",
                "UnitPrice": "source.UnitPrice",
                "ExpectedStockValue": "source.ExpectedStockValue",
                "ActualCashAmount": "source.ActualCashAmount",
                "VarianceAmount": "source.VarianceAmount",
                "SettlementLagDays": "source.SettlementLagDays",
                "ReconciliationStatus": "source.ReconciliationStatus",
                "GoldProcessedDate": F.current_timestamp()
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merge execution completed successfully.") 

###### Checkc the Stocl Cash Recon

In [0]:
%sql
SELECT * FROM glrecon.gold.gold_fact_stock_cash_recon ORDER BY StockTransactionId, CashTransactionId